# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a complete guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset title: ", metadata.name)
print("Description: ", metadata.description)
print("Version: ", getattr(metadata, 'version', 'N/A'))
print("Keywords: ", getattr(metadata, 'keywords', []))
print("License: ", getattr(metadata, 'license', ''))


## 2. Data Overview
Let's review the available record sets, their fields, and all corresponding `@id` values, following the Croissant schema structure. Entities are always referenced by their `@id`.

Below, we'll extract and list all available record sets and their information.

In [ ]:
print("Listing Record Sets:")
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the top-level metadata.")

for rs in record_sets:
    print(f"\nRecord Set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    {field.name} (@id: {field.id}, type: {field.data_type if hasattr(field, 'data_type') else 'N/A'})")
    if hasattr(rs, 'columns') and rs.columns is not None:
        print("  Columns:")
        for col in rs.columns:
            print(f"    {col.name} (@id: {col.id}, type: {col.data_type if hasattr(col, 'data_type') else 'N/A'})")


## 2.1. Example: Enumerate all record sets IDs
For programmatic extraction of `@id` for each record set:

In [ ]:
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("RecordSet @id values:", record_set_ids)

## 2.2. Preview Records from the First Record Set
Let's print a few records from the first record set using its `@id`. All record and field references use `@id`.

In [ ]:
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Previewing first 2 records from record set: {first_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=first_record_set_id)):
        print(rec)
        if i >= 1:
            break
else:
    print("No record set IDs found.")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis, referencing all sets by `@id`.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print("  Columns: ", dataframes[record_set_id].columns.tolist())
    print(dataframes[record_set_id].head(2))

## 4. Exploratory Data Analysis (EDA)
Here, we select a numeric field (referenced by its field `@id`) for analysis. For demonstration, we'll use the first record set and try to select a suitable numeric field (e.g. mass, coverage, counts, etc.). Fields/columns must be referenced by `@id`, and all filtering and grouping operations will use these IDs.

In [ ]:
# For simplicity, select the first record set and look for numeric fields
import numpy as np
from pandas.api.types import is_numeric_dtype

# Select which record set to analyze
rsid = record_set_ids[0]
df = dataframes[rsid]

# Identify candidate numeric fields (referenced by their @id)
numeric_ids = [c for c in df.columns if is_numeric_dtype(df[c])]
print('Numeric field IDs:', numeric_ids)
if not numeric_ids:
    # Try to cast columns to numeric if possible
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_ids = [c for c in df.columns if is_numeric_dtype(df[c])]

if numeric_ids:
    numeric_field_id = numeric_ids[0]
    # Pick a threshold value - 10 is arbitrary; use quantiles for more dynamic EDA
    values = df[numeric_field_id]
    threshold = np.nanquantile(values, 0.25)  # 25th percentile as threshold

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (25th percentile):")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group-by field (string/categorical, not the index)
    non_numeric = [c for c in df.columns if not is_numeric_dtype(df[c])]
    group_field_id = non_numeric[0] if non_numeric else None
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field for group-by found.")
else:
    print('No numeric columns found for EDA.')

## 5. Visualization
Visualize the distribution of the selected numeric field or its relationship with other fields, using `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_ids:
    field = numeric_field_id
    plt.figure(figsize=(6, 4))
    sns.histplot(df[field], kde=True)
    plt.xlabel(field)
    plt.title(f"Distribution of {field}")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[field])
        plt.xlabel(group_field_id)
        plt.ylabel(field)
        plt.title(f"{field} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we have:

- Loaded the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) Croissant-annotated dataset and explored its metadata and structure.
- Listed and examined all available `RecordSet` objects, referencing all data entities by their Croissant `@id` field as required.
- Loaded record set data into pandas DataFrames for flexible analysis.
- Demonstrated basic filtering, normalization, and grouping, and visualized field distributions.

**Next steps:**
- Deeper analysis of specific protein quantification or metadata columns using their respective `@id` values.
- Integration with ML workflows using `mlcroissant`'s streaming and schema utilities.
- Exploration of additional record sets and complex relationships among experimental conditions if present.

Be sure to consult the [documentation](https://mlcommons.github.io/croissant/) and the Croissant schema for full field meanings.